In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import boto3
import datetime as dt
from tqdm import tqdm
import shutil

try:
    import optbinning
except:
    ! pip install optbinning

try:
    import catboost
except:
    ! pip install catboost

In [ ]:
dtm_now = dt.datetime.now()
print(f'Latest run date: {dtm_now}')

#### Constants

In [ ]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# output
str_dirname_output = './output'

#### Make output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import raw data

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/04_join_targets/{str_filename}'
df = pd.read_parquet(
    str_uri,
)
# show
df

#### Get Gen 12 predictions

In [ ]:
list_cols = [
    'gen12_pd',
    'gen12_lgd',
]
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/05_get_gen12_predictions/{str_filename}'
df_tmp = pd.read_parquet(
    str_uri,
    columns=list_cols,
)
# show
df_tmp

#### Concat horizontally

In [ ]:
df = pd.concat([df, df_tmp], axis=1)
df

#### Get Gen 13 Predictions

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/06_get_gen13_predictions/{str_filename}'
df_tmp = pd.read_parquet(
    str_uri,
)
# get columns we want
list_cols = [col for col in df_tmp.columns if '_contribution' in col]
list_cols = list_cols + ['gen13_pd', 'gen13_lgd']
# subset
df_tmp = df_tmp[list_cols].copy()

# show
df_tmp

#### Concat horizontally

In [ ]:
df = pd.concat([df, df_tmp], axis=1)
df

#### Save to s3

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)